In [1]:
%pip install rdkit

   ---------------------------------------- 0.0/24.5 MB ? eta -:--:--
   --- ------------------------------------ 2.1/24.5 MB 9.8 MB/s eta 0:00:03
   ----------- ---------------------------- 6.8/24.5 MB 16.8 MB/s eta 0:00:02
   -------------------- ------------------- 12.6/24.5 MB 20.2 MB/s eta 0:00:01
   ----------------------- ---------------- 14.4/24.5 MB 17.4 MB/s eta 0:00:01
   -------------------------- ------------- 16.5/24.5 MB 16.0 MB/s eta 0:00:01
   ------------------------------ --------- 18.6/24.5 MB 14.9 MB/s eta 0:00:01
   --------------------------------- ------ 20.4/24.5 MB 14.0 MB/s eta 0:00:01
   ----------------------------------- ---- 21.5/24.5 MB 12.9 MB/s eta 0:00:01
   ------------------------------------ --- 22.5/24.5 MB 12.0 MB/s eta 0:00:01
   -------------------------------------- - 23.6/24.5 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------  24.4/24.5 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------- 24.5/24.5 MB 9.8 MB/

In [26]:
import pandas as pd
import numpy as np
import pickle
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm import tqdm

# =========================
# CONFIG
# =========================
CSV_PATH = "final_processed_data.csv"
COL1 = "SMILES1"
COL2 = "SMILES2"

RADIUS = 2       # Morgan radius (2 = ECFP4, 3 = ECFP6)
N_BITS = 256      # Embedding dimension (match your old EMBED_DIM)
CACHE_PATH = "drug_embeddings.pkl"


# =========================
# STEP 1: SMILES → Morgan Fingerprint
# =========================
def smiles_to_morgan(smiles, radius=RADIUS, n_bits=N_BITS):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(n_bits, dtype=np.float32)

    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp, dtype=np.float32)


# =========================
# STEP 2: Load dataset
# =========================
df = pd.read_csv(CSV_PATH)
all_smiles = set(df[COL1]).union(set(df[COL2]))

print(f"Total unique drugs: {len(all_smiles)}")


# =========================
# STEP 3: Compute embeddings
# =========================
embedding_cache = {}

for sm in tqdm(all_smiles, desc="Computing Morgan fingerprints"):
    embedding_cache[sm] = smiles_to_morgan(sm)


# =========================
# STEP 4: Save embeddings
# =========================
with open(CACHE_PATH, "wb") as f:
    pickle.dump(embedding_cache, f)

print(f"Embeddings saved to '{CACHE_PATH}'")


# =========================
# STEP 5: Sanity check
# =========================
from sklearn.metrics.pairwise import cosine_similarity

embs = np.stack(list(embedding_cache.values()))  # [N, 256]

# Sample 500 random pairs for speed
idx = np.random.choice(len(embs), size=min(500, len(embs)), replace=False)
sample = embs[idx]

sims = cosine_similarity(sample)
upper = sims[np.triu_indices_from(sims, k=1)]  # strictly upper triangle

norms = np.linalg.norm(embs, axis=1)

print("\n--- Embedding Diagnostics ---")
print(f"  Shape          : {embs.shape}")
print(f"  Mean norm      : {norms.mean():.4f}  (expect ~3–15 for bit vectors)")
print(f"  Std norm       : {norms.std():.4f}")
print(f"  Cosine mean    : {upper.mean():.4f}  (expect < 0.5 for diverse drugs)")
print(f"  Cosine std     : {upper.std():.4f}  (expect > 0.2)")
print(f"  Min cosine     : {upper.min():.4f}")
print(f"  Max cosine     : {upper.max():.4f}")

Total unique drugs: 644


Computing Morgan fingerprints:   0%|          | 0/644 [00:00<?, ?it/s][17:41:23] DEPRECATION WARNING: please use MorganGenerator
[17:41:23] DEPRECATION WARNING: please use MorganGenerator
[17:41:23] DEPRECATION WARNING: please use MorganGenerator
[17:41:23] DEPRECATION WARNING: please use MorganGenerator
[17:41:23] DEPRECATION WARNING: please use MorganGenerator
[17:41:23] DEPRECATION WARNING: please use MorganGenerator
[17:41:23] DEPRECATION WARNING: please use MorganGenerator
[17:41:23] DEPRECATION WARNING: please use MorganGenerator
[17:41:23] DEPRECATION WARNING: please use MorganGenerator
[17:41:23] DEPRECATION WARNING: please use MorganGenerator
[17:41:23] DEPRECATION WARNING: please use MorganGenerator
[17:41:23] DEPRECATION WARNING: please use MorganGenerator
[17:41:23] DEPRECATION WARNING: please use MorganGenerator
[17:41:23] DEPRECATION WARNING: please use MorganGenerator
[17:41:23] DEPRECATION WARNING: please use MorganGenerator
[17:41:23] DEPRECATION WARNING: please use Mo

Embeddings saved to 'drug_embeddings.pkl'

--- Embedding Diagnostics ---
  Shape          : (644, 256)
  Mean norm      : 6.2922  (expect ~3–15 for bit vectors)
  Std norm       : 1.3224
  Cosine mean    : 0.2726  (expect < 0.5 for diverse drugs)
  Cosine std     : 0.0948  (expect > 0.2)
  Min cosine     : 0.0000
  Max cosine     : 0.9903


In [ ]:
# how many bits are ON per molecule
bit_counts = embs.sum(axis=1)

print("Avg bits ON:", bit_counts.mean())
print("Std bits ON:", bit_counts.std())
print("Min bits ON:", bit_counts.min())
print("Max bits ON:", bit_counts.max())


Embedding Statistics:
num_samples: 644
dim: 256
mean_norm: 6.292169570922852
std_norm: 1.3233991861343384
cosine_mean: 0.27142202854156494
cosine_std: 0.09503525495529175


In [42]:
# how many bits are ON per molecule
bit_counts = embs.sum(axis=1)

print("Avg bits ON:", bit_counts.mean())
print("Std bits ON:", bit_counts.std())
print("Min bits ON:", bit_counts.min())
print("Max bits ON:", bit_counts.max())

Avg bits ON: 41.34006
Std bits ON: 16.105268
Min bits ON: 1.0
Max bits ON: 114.0


In [43]:
from rdkit.DataStructs import TanimotoSimilarity
from rdkit import DataStructs

def fp_from_array(arr):
    fp = DataStructs.ExplicitBitVect(len(arr))
    for i, v in enumerate(arr):
        if v == 1:
            fp.SetBit(i)
    return fp

fps = [fp_from_array(x) for x in embs[:500]]  # sample

sims = []
for i in range(len(fps)):
    for j in range(i+1, len(fps)):
        sims.append(TanimotoSimilarity(fps[i], fps[j]))

sims = np.array(sims)

print("\n--- Tanimoto Stats ---")
print("Mean:", sims.mean())
print("Std:", sims.std())
print("Min:", sims.min())
print("Max:", sims.max())


--- Tanimoto Stats ---
Mean: 0.15378419657851397
Std: 0.0651524367672884
Min: 0.0
Max: 0.9807692307692307


In [44]:
unique_rows = np.unique(embs, axis=0)
print("Unique embeddings:", len(unique_rows))
print("Total embeddings:", len(embs))

Unique embeddings: 644
Total embeddings: 644
